[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_18/MFM_ch18_seminar/MFM_ch18_seminar.ipynb)

# MFM_ch18_seminar

Seminar 18 computations: CoVaR and MES under the bivariate Normal distribution, SRISK and break-even leverage, connectedness tables, reverse stress tests, Delta-CoVaR of US and European banks with block and i.i.d. bootstrap intervals, VaR versus Delta-CoVaR rank correlation, robustness of the Diebold-Yilmaz index, a Newey-West test on the rolling index, the FRM penalty in one window, a stress test of a Romanian bank portfolio and the exposure of Romanian banks to euro-area banking stress; charts for the solutions (conditional densities behind Delta-CoVaR, the reverse stress geometry, crisis-path loss distributions of LRMES, the FRM coefficients, the Kupiec acceptance region and power).

*Modelling Financial Markets (MFM), Chapter 18: Systemic Risk and Networks. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_18'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from scipy.stats import spearmanr
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
from sklearn.linear_model import QuantileRegressor, ElasticNetCV
from scipy import optimize
try:
    from arch import arch_model
except ImportError:                      # Colab: pachetul arch nu este preinstalat
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])
    from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Teal     = '#17A2B8'
Gray     = '#7F7F7F'   # doar linii de referinta, benzi, grila
REGION_COL = {'US': MainBlue, 'EU': IDAred, 'RO': Forest}
REGION_LAB = {'US': 'US banks', 'EU': 'European banks', 'RO': 'Romanian banks (BVB)'}
EVENTS = [('2011-08-08', 'Euro crisis'), ('2016-06-24', 'Brexit vote'), ('2018-12-19', 'RO bank tax'),
          ('2020-03-16', 'COVID-19'), ('2023-03-13', 'SVB'), ('2025-04-07', 'Tariffs')]

HERE = '.'
SEED = 42
B_BOOT = 999
RESULTS = {}


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(x):
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return [jsonable(v) for v in x.tolist()]
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, pd.Timestamp):
        return str(x.date())
    return x


def mark_events(ax, ymax=None):
    """Linii verticale de referinta pentru evenimente, cu eticheta neagra."""
    for d, lab in EVENTS:
        t = pd.Timestamp(d)
        if ax.get_xlim()[0] <= mdates.date2num(t) <= ax.get_xlim()[1]:
            ax.axvline(t, color=Gray, lw=0.5, ls=':')
            ax.text(t, ax.get_ylim()[1] if ymax is None else ymax, lab, rotation=90, fontsize=6.5, color='black',
                    ha='right', va='top')

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
START = '2010-01-01'
END = '2026-09-18'

# banca -> (simbol, nume, regiune)
BANKS = {
    'JPM':  ('JPM.US',    'JPMorgan Chase',       'US'),
    'BAC':  ('BAC.US',    'Bank of America',      'US'),
    'C':    ('C.US',      'Citigroup',            'US'),
    'GS':   ('GS.US',     'Goldman Sachs',        'US'),
    'MS':   ('MS.US',     'Morgan Stanley',       'US'),
    'WFC':  ('WFC.US',    'Wells Fargo',          'US'),
    'DBK':  ('DBK.XETRA', 'Deutsche Bank',        'EU'),
    'BNP':  ('BNP.PA',    'BNP Paribas',          'EU'),
    'SAN':  ('SAN.MC',    'Santander',            'EU'),
    'INGA': ('INGA.AS',   'ING',                  'EU'),
    'HSBA': ('HSBA.LSE',  'HSBC',                 'EU'),
    'TLV':  ('TLV.RO',    'Banca Transilvania',   'RO'),
    'BRD':  ('BRD.RO',    'BRD-Groupe SG',        'RO'),
}
US = [k for k, v in BANKS.items() if v[2] == 'US']
EU = [k for k, v in BANKS.items() if v[2] == 'EU']
RO = [k for k, v in BANKS.items() if v[2] == 'RO']
ALL = US + EU + RO
NAMES = {k: v[1] for k, v in BANKS.items()}
INDICES = {'SPX': ('GSPC.INDX', 'S&P 500'), 'SX5E': ('STOXX50E.INDX', 'Euro Stoxx 50'),
           'BET': ('BET', 'BET'), 'VIX': ('VIX.INDX', 'VIX')}
REGION_INDEX = {'US': 'SPX', 'EU': 'SX5E', 'RO': 'BET'}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def bank_frame(key, start=START, end=END):
    """OHLCV al unei banci, cu pretul ajustat si corectiile de mai sus."""
    symbol, _, region = BANKS[key]
    d = read_market(symbol).loc[start:end].copy()
    d = d[(d.index.dayofweek < 5) & (d['adjusted_close'] > 0)]
    if region == 'RO':
        d = d[d['volume'] > 0]                                   # zile fara tranzactii
    if key == 'TLV' and pd.Timestamp('2016-05-30') in d.index:
        # ajustarea pentru actiunile gratuite: factorul de dupa data ex se aplica si pe 30 mai 2016
        f = d.loc['2016-05-31', 'adjusted_close'] / d.loc['2016-05-31', 'close']
        d.loc['2016-05-30', 'adjusted_close'] = d.loc['2016-05-30', 'close'] * f
    return d


def bank_price(key, start=START, end=END):
    return bank_frame(key, start, end)['adjusted_close'].rename(key)


def index_close(key, start=START, end=END):
    s = read_market(INDICES[key][0])['close'].loc[start:end]
    s = s[(s.index.dayofweek < 5) & (s > 0)].dropna()
    if key != 'VIX':
        s = s[s.diff() != 0]                                     # sarbatori completate cu pretul anterior
    return s.rename(key)


def prices(keys, start=START, end=END):
    """Preturi zilnice pentru banci si/sau indici, doar in zilele comune."""
    cols = [bank_price(k, start, end) if k in BANKS else index_close(k, start, end) for k in keys]
    return pd.concat(cols, axis=1, join='inner').dropna()


def joint_returns(keys, start=START, end=END):
    """Randamente log zilnice in %: intai join pe PRETURI in zilele comune, apoi randamente."""
    return 100 * np.log(prices(keys, start, end)).diff().dropna()


def bank_range_vol(keys, start=START, end=END):
    """Volatilitatea zilnica Parkinson (in %, anualizata), in zilele comune tuturor bancilor."""
    out = []
    for k in keys:
        d = bank_frame(k, start, end)
        rng = np.log(d['high'] / d['low']).where(d['high'] > d['low'])
        s2 = rng ** 2 / (4 * np.log(2))
        floor = s2.where(s2 > 0).cummin().ffill()                  # cea mai mica valoare pozitiva observata PANA la ziua t
        s2 = s2.fillna(floor)                                     # zile cu maxim = minim: fara informatie din viitor
        out.append((100 * np.sqrt(252 * s2)).rename(k))
    return pd.concat(out, axis=1, join='inner').dropna()


def system_return(R, members):
    """Randamentul sistemului: portofoliul cu ponderi egale al bancilor (randamente log in %, aproximare zilnica)."""
    return R[members].mean(axis=1)


# randamente log zilnice in % (zile comune) si volatilitatea Parkinson (log)
R = joint_returns(ALL + ['SPX', 'SX5E', 'BET', 'VIX'])
V = np.log(bank_range_vol(ALL))
print(R.index[0].date(), '->', R.index[-1].date(), len(R), 'common days')


def system_ex(k, members):
    """Randamentul sistemului regional fara banca k (portofoliu cu ponderi egale)."""
    return R[[j for j in members if j != k]].mean(axis=1)


def region_of(k):
    return BANKS[k][2]

## Systemic risk measures

In [ ]:
def mes(r_i, r_m, alpha=0.05):
    """MES (%): pierderea medie a bancii in zilele in care piata este in coada de probabilitate alpha."""
    q = np.quantile(r_m, alpha)
    return -np.mean(np.asarray(r_i)[np.asarray(r_m) <= q])


def mes_threshold(r_i, r_m, c=-2.0):
    """MES (%) in zilele in care piata scade cu mai mult de |c|% (definitia folosita pentru LRMES)."""
    return -np.mean(np.asarray(r_i)[np.asarray(r_m) < c])


def lrmes(mes2_pct):
    """Pierderea pe termen lung intr-o criza (piata -40% in sase luni): LRMES = 1 - exp(-18 MES), MES in fractii."""
    return 1 - np.exp(-18 * mes2_pct / 100)


def srisk_ratio(lrm, leverage, k=0.08):
    """SRISK / capitalul de piata W: k(L - 1) - (1 - k)(1 - LRMES), cu L = (D + W)/W."""
    return k * (leverage - 1) - (1 - k) * (1 - lrm)


def breakeven_leverage(lrm, k=0.08):
    """Levierul L* peste care SRISK > 0: L* = 1 + (1 - k)(1 - LRMES)/k."""
    return 1 + (1 - k) * (1 - lrm) / k


def qreg(y, X, q):
    """Regresie cuantila (Koenker-Bassett); intoarce coeficientii [termen liber, pante]."""
    return QuantReg(np.asarray(y), sm.add_constant(np.asarray(X), has_constant='add')).fit(q=q, max_iter=5000).params


def covar_static(x_sys, x_i, alpha=0.01):
    """CoVaR si Delta-CoVaR necondiționate (%, pozitive = pierderi).

    Regresia cuantila la nivelul alpha: X_sys = a + b X_i;
    CoVaR_alpha = -(a + b q_alpha(X_i)); Delta-CoVaR = b (q_50(X_i) - q_alpha(X_i))."""
    x_sys, x_i = np.asarray(x_sys), np.asarray(x_i)
    a, b = qreg(x_sys, x_i, alpha)
    qa, qm = np.quantile(x_i, alpha), np.quantile(x_i, 0.5)
    return {'a': a, 'b': b, 'var_i': -qa, 'covar': -(a + b * qa), 'covar_med': -(a + b * qm),
            'dcovar': b * (qm - qa), 'var_sys': -np.quantile(x_sys, alpha)}


def block_bootstrap_idx(n, block=20, rng=None):
    """Indici pentru bootstrap pe blocuri mobile (Künsch), blocuri de lungime fixa."""
    rng = rng or np.random.default_rng(0)
    starts = rng.integers(0, n - block + 1, size=int(np.ceil(n / block)))
    return (starts[:, None] + np.arange(block)[None, :]).ravel()[:n]


def covar_boot(x_sys, x_i, alpha=0.01, B=200, block=20, seed=0):
    """Interval percentil de 95% pentru Delta-CoVaR prin bootstrap pe blocuri mobile."""
    x_sys, x_i = np.asarray(x_sys), np.asarray(x_i)
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(B):
        idx = block_bootstrap_idx(len(x_i), block, rng)
        d.append(covar_static(x_sys[idx], x_i[idx], alpha)['dcovar'])
    return np.percentile(d, [2.5, 97.5])


def covar_dynamic(x_sys, x_i, M, alpha=0.01):
    """Delta-CoVaR variabil in timp cu variabile de stare M_{t-1} (Adrian & Brunnermeier, 2016).

    X_i,t = a + c'M_{t-1} la nivelurile alpha si 50%  ->  VaR_i,t;
    X_sys,t = a + b X_i,t + c'M_{t-1} la nivelul alpha  ->  Delta-CoVaR_t = b (q50_i,t - qalpha_i,t)."""
    ci_a = qreg(x_i, M, alpha)
    ci_m = qreg(x_i, M, 0.5)
    cs = qreg(x_sys, np.column_stack([x_i, M]), alpha)
    Mc = sm.add_constant(np.asarray(M), has_constant='add')
    qa, qm = Mc @ ci_a, Mc @ ci_m
    return pd.DataFrame({'var_i': -qa, 'dcovar': cs[1] * (qm - qa)}, index=getattr(x_i, 'index', None)), cs[1]


def var_fit(Y, p):
    """VAR(p) estimat prin OLS; intoarce coeficientii A_1..A_p (k x k) si matricea de covarianta a reziduurilor."""
    Y = np.asarray(Y, float)
    T, k = Y.shape
    X = np.hstack([np.ones((T - p, 1))] + [Y[p - l - 1:T - l - 1] for l in range(p)])
    Yt = Y[p:]
    B = np.linalg.lstsq(X, Yt, rcond=None)[0]
    E = Yt - X @ B
    S = E.T @ E / (len(Yt) - X.shape[1])
    A = [B[1 + l * k:1 + (l + 1) * k].T for l in range(p)]
    return A, S


def var_bic(Y, pmax=5):
    """Ordinul VAR ales prin criteriul BIC (Schwarz), cu matricea de covarianta a reziduurilor de verosimilitate maxima."""
    Y = np.asarray(Y, float)
    T, k = Y.shape
    best = None
    n = T - pmax
    for p in range(1, pmax + 1):
        A, S = var_fit(Y[pmax - p:], p)
        S_ml = S * (n - 1 - p * k) / n                        # covarianta de verosimilitate maxima E'E / n
        bic = np.log(np.linalg.det(S_ml)) + np.log(n) * p * k * k / n
        if best is None or bic < best[1]:
            best = (p, bic)
    return best[0]


def ma_coefs(A, H):
    """Coeficientii reprezentarii medie mobila Phi_0..Phi_{H-1}."""
    k = A[0].shape[0]
    Phi = [np.eye(k)]
    for h in range(1, H):
        Phi.append(sum(A[l] @ Phi[h - l - 1] for l in range(min(h, len(A)))))
    return Phi


def gfevd(A, S, H=10):
    """Descompunerea generalizata a dispersiei erorii de prognoza (Pesaran-Shin), normalizata pe linii."""
    Phi = ma_coefs(A, H)
    k = S.shape[0]
    num = np.zeros((k, k))
    den = np.zeros(k)
    for P in Phi:
        PS = P @ S
        num += PS ** 2
        den += np.diag(P @ S @ P.T)
    theta = num / np.diag(S)[None, :] / den[:, None]
    return theta / theta.sum(axis=1, keepdims=True)


def spillover_table(theta, names):
    """Tabelul de conectivitate: contributii (%), FROM, TO, NET si indicele total."""
    k = theta.shape[0]
    D = 100 * theta
    off = D - np.diag(np.diag(D))
    frm, to = off.sum(axis=1), off.sum(axis=0)
    tab = pd.DataFrame(D, index=names, columns=names)
    return {'table': tab, 'from': pd.Series(frm, index=names), 'to': pd.Series(to, index=names),
            'net': pd.Series(to - frm, index=names), 'total': off.sum() / k,
            'pairwise_net': pd.DataFrame(off.T - off, index=names, columns=names)}


def rolling_total(Y, window=250, step=5, p=2, H=10):
    """Indicele total de conectivitate pe ferestre mobile."""
    Y = pd.DataFrame(Y)
    out, frm_to = {}, {}
    for end in range(window, len(Y) + 1, step):
        w = Y.iloc[end - window:end]
        A, S = var_fit(w.values, p)
        st = spillover_table(gfevd(A, S, H), list(Y.columns))
        out[Y.index[end - 1]] = st['total']
        frm_to[Y.index[end - 1]] = st['net']
    return pd.Series(out), pd.DataFrame(frm_to).T


def check_loss(u, tau):
    return u * (tau - (u < 0))


def lasso_qr_gacv(y, X, tau=0.05, lambdas=None):
    """Regresie cuantila penalizata L1; lambda ales prin GACV = sum rho_tau(rez) / (n - df)."""
    y, X = np.asarray(y, float), np.asarray(X, float)
    n = len(y)
    lambdas = np.logspace(-4, -0.5, 15) if lambdas is None else lambdas
    rows = []
    for lam in lambdas:
        m = QuantileRegressor(quantile=tau, alpha=lam, solver='highs').fit(X, y)
        res = y - m.predict(X)
        df = int(np.sum(np.abs(m.coef_) > 1e-8))
        g = check_loss(res, tau).sum() / max(n - df, 1)
        rows.append((lam, g, df, m.coef_.copy()))
    i = int(np.argmin([r[1] for r in rows]))
    return {'lambda': rows[i][0], 'gacv': [r[1] for r in rows], 'df': [r[2] for r in rows],
            'lambdas': list(lambdas), 'coef': rows[i][3], 'df_sel': rows[i][2]}

## Helper functions

In [ ]:
B = 999

MACRO = ['SPX', 'SX5E', 'VIX']

LAMBDAS = np.logspace(-6, -1.5, 20)

A5_M = np.array([[70.0, 20.0, 10.0], [25.0, 60.0, 15.0], [5.0, 10.0, 85.0]], float)

A6_M = np.array([[55.0, 25.0, 15.0, 5.0], [20.0, 50.0, 20.0, 10.0], [10.0, 15.0, 65.0, 10.0], [5.0, 5.0, 10.0, 80.0]], float)

def frm_design():
    """Randamente zilnice (fractii) ale bancilor si variabilele macro intarziate cu o zi."""
    Rf = R / 100
    Mac = Rf[MACRO].shift(1).add_suffix('_lag')
    return Rf[ALL].join(Mac).dropna()


def frm_window(D, k, tau=0.05):
    X = D[[j for j in ALL if j != k] + [c for c in D.columns if c.endswith('_lag')]]
    return lasso_qr_gacv(D[k].values, X.values, tau, LAMBDAS)


def weekly_returns(keys):
    p = prices(keys)
    w = p.resample('W-FRI').last().dropna()
    return 100 * np.log(w).diff().dropna()


def spill_rolling():
    tot, net = rolling_total(V, window=250, step=5, p=2, H=10)
    return tot, net


def var_simulate(A, c, E, Y0, idx):
    p = len(A)
    T = len(idx)
    k = A[0].shape[0]
    Y = np.zeros((T + p, k))
    Y[:p] = Y0
    for t in range(p, T + p):
        Y[t] = c + sum(A[l] @ Y[t - l - 1] for l in range(p)) + E[idx[t - p]]
    return Y


def var_fit_c(Y, p):
    Y = np.asarray(Y, float)
    T, k = Y.shape
    X = np.hstack([np.ones((T - p, 1))] + [Y[p - l - 1:T - l - 1] for l in range(p)])
    Bc = np.linalg.lstsq(X, Y[p:], rcond=None)[0]
    E = Y[p:] - X @ Bc
    return [Bc[1 + l * k:1 + (l + 1) * k].T for l in range(p)], Bc[0], E


def spill_bootstrap(Y, p, H=10, B=B, block=20, seed=SEED):
    """Interval percentil pentru conectivitatea totala si neta (bootstrap pe blocuri de reziduuri, VAR reestimat)."""
    Y = np.asarray(Y, float)
    A, c, E = var_fit_c(Y, p)
    rng = np.random.default_rng(seed)
    tot, net = [], []
    for _ in range(B):
        idx = block_bootstrap_idx(len(E), block, rng)
        Yb = var_simulate(A, c, E, Y[:p], idx)
        Ab, Sb = var_fit(Yb, p)
        st = spillover_table(gfevd(Ab, Sb, H), list(range(Y.shape[1])))
        tot.append(st['total'])
        net.append(st['net'].values)
    return np.array(tot), np.array(net)


def kupiec(x, n, p):
    """Testul Kupiec (acoperire neconditionata): LR ~ chi2(1) sub H0: rata de depasire = p."""
    ph = x / n
    ll0 = x * np.log(p) + (n - x) * np.log(1 - p)
    ll1 = (x * np.log(ph) if x > 0 else 0) + ((n - x) * np.log(1 - ph) if x < n else 0)
    lr = -2 * (ll0 - ll1)
    return lr, stats.chi2.sf(lr, 1)


def backtest_ge(alpha=0.05, start=1000):
    """Backtest in afara esantionului al CoVaR Girardi-Ergun (istoric, fereastra extinsa):
    lovitura comuna 1{X_sys <= -CoVaR_t, X_i <= -VaR_i,t}, cu probabilitatea alpha^2 sub H0."""
    out = {}
    for k in US + EU:
        xs = system_ex(k, US if k in US else EU).values
        xi = R[k].values
        hits, vh = [], []
        for t in range(start, len(xi)):
            a, s = xi[:t], xs[:t]
            q = np.quantile(a, alpha)
            cov = -np.quantile(s[a <= q], alpha)
            vh.append(xi[t] <= q)
            hits.append((xi[t] <= q) and (xs[t] <= -cov))
        n, x = len(hits), int(np.sum(hits))
        lr, p = kupiec(x, n, alpha ** 2)
        lrv, pv = kupiec(int(np.sum(vh)), n, alpha)
        out[k] = {'n': n, 'hits': x, 'expected': n * alpha ** 2, 'lr': lr, 'p': p,
                  'var_hits': int(np.sum(vh)), 'var_p': pv}
    return out


def gjr_fit(r):
    """GJR-GARCH(1,1) cu medie zero (randamente log in %), QML."""
    m = arch_model(r, mean='Zero', vol='GARCH', p=1, o=1, q=1, dist='normal', rescale=False).fit(disp='off')
    om, a, g, b = m.params[['omega', 'alpha[1]', 'gamma[1]', 'beta[1]']]
    s2 = m.conditional_volatility ** 2
    s2_next = om + (a + g * (r[-1] < 0)) * r[-1] ** 2 + b * s2[-1]
    return {'omega': om, 'alpha': a, 'gamma': g, 'beta': b, 'sig': np.sqrt(s2), 's2_next': s2_next}


def dcc_fit(e):
    """DCC(1,1) bivariat, al doilea pas al QML (Engle, 2002); e = reziduuri standardizate T x 2."""
    S = np.corrcoef(e.T)
    T = len(e)

    def filt(a, b):
        Q = S.copy()
        rho = np.empty(T)
        for t in range(T):
            rho[t] = Q[0, 1] / np.sqrt(Q[0, 0] * Q[1, 1])
            Q = (1 - a - b) * S + a * np.outer(e[t], e[t]) + b * Q
        return rho, Q

    def nll(th):
        a, b = th
        if a <= 0 or b <= 0 or a + b >= 0.999:
            return 1e10
        rho, _ = filt(a, b)
        z1, z2 = e[:, 0], e[:, 1]
        return 0.5 * np.sum(np.log(1 - rho ** 2) + (z1 ** 2 + z2 ** 2 - 2 * rho * z1 * z2) / (1 - rho ** 2)
                            - (z1 ** 2 + z2 ** 2))

    best = min((optimize.minimize(nll, x0, method='Nelder-Mead', options={'xatol': 1e-5, 'fatol': 1e-4})
                for x0 in [(0.02, 0.95), (0.05, 0.90)]), key=lambda o: o.fun)
    a, b = best.x
    rho, Q = filt(a, b)
    return {'a': a, 'b': b, 'S': S, 'rho': rho, 'Q_next': Q}


def simulate_market(gm, eps_m, idx):
    """Traiectorii ale pietei (GJR) din inovatiile reesantionate eps_m[idx], idx: S x h."""
    Sn, h = idx.shape
    s2 = np.full(Sn, gm['s2_next'])
    cum = np.zeros(Sn)
    for t in range(h):
        r = np.sqrt(s2) * eps_m[idx[:, t]]
        cum += r
        s2 = gm['omega'] + (gm['alpha'] + gm['gamma'] * (r < 0)) * r ** 2 + gm['beta'] * s2
    return cum


def simulate_firm(gf, gm, dcc, eps_m, xi, idx):
    """Traiectoriile bancii conditionate pe aceleasi date extrase (Anexa A din Brownlees & Engle, 2017)."""
    Sn, h = idx.shape
    s2i = np.full(Sn, gf['s2_next'])
    q11 = np.full(Sn, dcc['Q_next'][0, 0])
    q22 = np.full(Sn, dcc['Q_next'][1, 1])
    q12 = np.full(Sn, dcc['Q_next'][0, 1])
    cum = np.zeros(Sn)
    a, b, S = dcc['a'], dcc['b'], dcc['S']
    for t in range(h):
        rho = q12 / np.sqrt(q11 * q22)
        em = eps_m[idx[:, t]]
        ei = rho * em + np.sqrt(1 - rho ** 2) * xi[idx[:, t]]
        r = np.sqrt(s2i) * ei
        cum += r
        s2i = gf['omega'] + (gf['alpha'] + gf['gamma'] * (r < 0)) * r ** 2 + gf['beta'] * s2i
        q11 = (1 - a - b) * S[0, 0] + a * ei ** 2 + b * q11
        q22 = (1 - a - b) * S[1, 1] + a * em ** 2 + b * q22
        q12 = (1 - a - b) * S[0, 1] + a * ei * em + b * q12
    return cum

## Analysis

In [ ]:
def a1_covar_normal(s_sys=1.5, s_i=2.0, rho=0.6, alpha=0.01):
    """CoVaR sub distributia Normala bivariata cu medii zero."""
    z = stats.norm.ppf(alpha)
    return {'z': z, 'var_i': -s_i * z, 'var_sys': -s_sys * z, 'cond_sd': s_sys * np.sqrt(1 - rho ** 2),
            'covar': -s_sys * z * (rho + np.sqrt(1 - rho ** 2)), 'covar_med': -s_sys * np.sqrt(1 - rho ** 2) * z,
            'dcovar': -rho * s_sys * z}


def a2_covar_normal():
    """Doua banci: B are VaR dublu, dar corelatie mai mica cu sistemul."""
    return {'A': a1_covar_normal(rho=0.6, s_i=2.0), 'B': a1_covar_normal(rho=0.3, s_i=4.0)}


def a2_ge_covar(s_sys=1.5, rho=0.6, alpha=0.01):
    """CoVaR Girardi-Ergun sub distributia Normala: P(X_sys <= -c, X_i <= q_alpha(X_i)) = alpha^2."""
    from scipy.optimize import brentq
    z = stats.norm.ppf(alpha)
    mvn = stats.multivariate_normal(mean=[0, 0], cov=[[1, rho], [rho, 1]])
    f = lambda c: mvn.cdf([-c / s_sys, z]) - alpha ** 2
    c = brentq(f, 0.01, 20)
    ab = a1_covar_normal(s_sys=s_sys, rho=rho, alpha=alpha)
    return {'ge_covar': c, 'ab_covar': ab['covar'], 'ab_dcovar': ab['dcovar'], 'z': z}


def a3_euler(w=(0.5, 0.5), s=(2.0, 3.0), rho=0.5, alpha=0.05):
    """Descompunerea Euler a ES sub distributia Normala: ES_p = sum w_i MES_i, cu MES conditionat pe portofoliu."""
    w, s = np.asarray(w), np.asarray(s)
    C = np.array([[s[0] ** 2, rho * s[0] * s[1]], [rho * s[0] * s[1], s[1] ** 2]])
    sp = np.sqrt(w @ C @ w)
    k = stats.norm.pdf(stats.norm.ppf(alpha)) / alpha
    beta = C @ w / sp ** 2
    mes_i = beta * sp * k
    return {'sp': sp, 'k': k, 'es_p': sp * k, 'beta': beta, 'mes': mes_i, 'sum': w @ mes_i, 'cov': C @ w}


def a3_mes_srisk(beta=1.3, s_m=1.0, alpha=0.05, W=100.0, D=1100.0, k=0.08):
    """MES, LRMES si SRISK pentru o banca ilustrativa, cu randamente Normale bivariate."""
    z = stats.norm.ppf(alpha)
    es_m = s_m * stats.norm.pdf(z) / alpha
    tail2 = s_m * stats.norm.pdf(2 / s_m) / stats.norm.cdf(-2 / s_m)     # E[-R_m | R_m < -2]
    mes2 = beta * tail2
    lr = lrmes(mes2)
    return {'z': z, 'es_m': es_m, 'mes': beta * es_m, 'tail2': tail2, 'mes2': mes2, 'lrmes': lr,
            'kD': k * D, 'eq': (1 - k) * (1 - lr) * W, 'srisk': k * D - (1 - k) * (1 - lr) * W,
            'L': (D + W) / W, 'Lstar': breakeven_leverage(lr, k)}


def a4_lrmes_shortcut(beta=1.3, d=0.40):
    """De unde vine 18: LRMES = 1 - (1 - d)^beta = 1 - exp(-kappa MES^{2%}), kappa = -ln(1 - d) / ES^{2%}(pietei)."""
    m = R['SPX'].values
    es2 = -m[m < -2].mean() / 100                           # pierderea medie a S&P 500 in zilele sub -2% (fractie)
    kappa = -np.log(1 - d) / es2
    return {'es2': 100 * es2, 'n2': int((m < -2).sum()), 'kappa': kappa, 'ln': -np.log(1 - d),
            'exact': 1 - (1 - d) ** beta, 'short18': 1 - np.exp(-18 * beta * es2), 'shortk': 1 - np.exp(-kappa * beta * es2),
            'norm_tail2': stats.norm.pdf(2) / stats.norm.cdf(-2)}


def a4_srisk_k():
    """Aceeasi banca cu k = 5.5% (prag prudential mai mic) si pragul de levier."""
    r8 = a3_mes_srisk()
    r55 = a3_mes_srisk(k=0.055)
    return {'k8': r8, 'k55': r55}


def dy_from_matrix(M):
    """Tabelul Diebold-Yilmaz dintr-o matrice de contributii (%, liniile insumeaza 100)."""
    off = M - np.diag(np.diag(M))
    frm, to = off.sum(1), off.sum(0)
    return {'from': frm, 'to': to, 'net': to - frm, 'total': off.sum() / len(M), 'pairwise': (off.T - off)}


def a5_dy():
    return dy_from_matrix(A5_M)


def a6_dy():
    return dy_from_matrix(A6_M)


def a5_gfevd(A=((0.5, 0.2), (0.1, 0.4)), S=((1.0, 0.5), (0.5, 1.0))):
    """GFEVD la H = 1 si 2 pentru un VAR(1) bivariat: nenormalizat, normalizat si Cholesky."""
    A, S = np.asarray(A), np.asarray(S)
    out = {}
    for H in (1, 2):
        Phi = ma_coefs([A], H)
        num = sum((P @ S) ** 2 for P in Phi) / np.diag(S)[None, :]
        den = np.array([sum((P @ S @ P.T)[i, i] for P in Phi) for i in range(2)])
        th = num / den[:, None]
        out[f'H{H}'] = {'raw': th, 'rowsum': th.sum(1), 'norm': th / th.sum(1, keepdims=True),
                        'C': 100 * (th / th.sum(1, keepdims=True))[[0, 1], [1, 0]].sum() / 2}
    L = np.linalg.cholesky(S)
    num = (L ** 2)
    out['chol_H1'] = num / num.sum(1, keepdims=True)
    return out


def a6_gfevd():
    return a5_gfevd(A=((0.5, 0.0), (0.4, 0.5)), S=((1.0, 0.3), (0.3, 1.0)))


def reverse_2f(b, S, target):
    b, S = np.asarray(b, float), np.asarray(S, float)
    sb2 = b @ S @ b
    f = -target * S @ b / sb2
    d = target / np.sqrt(sb2)
    return {'Sb': S @ b, 'sb2': sb2, 'sb': np.sqrt(sb2), 'f': f, 'd': d, 'p': stats.norm.cdf(-d)}


def a7_reverse():
    return reverse_2f([1.2, 0.8], [[16, 6], [6, 9]], 20)


def a8_reverse():
    """Scenariul A7 sub o distributie Student-t multivariata cu nu = 4 (aceeasi covarianta)."""
    r = reverse_2f([1.2, 0.8], [[16, 6], [6, 9]], 20)
    nu = 4
    out = {'nu': nu, 'd': r['d'], 'p_norm': r['p'], 'scale': np.sqrt(nu / (nu - 2)),
           'p_t': stats.t.sf(r['d'] * np.sqrt(nu / (nu - 2)), nu)}
    r30 = reverse_2f([1.2, 0.8], [[16, 6], [6, 9]], 30)
    out.update({'d30': r30['d'], 'p30_norm': r30['p'], 'p30_t': stats.t.sf(r30['d'] * np.sqrt(nu / (nu - 2)), nu)})
    out['ratio'] = out['p_t'] / out['p_norm']
    out['ratio30'] = out['p30_t'] / out['p30_norm']
    return out


def b1_us_dcovar5():
    """Delta-CoVaR 5% pentru bancile americane: interval bootstrap pe blocuri vs i.i.d."""
    out = {}
    rng = np.random.default_rng(SEED)
    for i, k in enumerate(US):
        xs, xi = system_ex(k, US).values, R[k].values
        c = covar_static(xs, xi, 0.05)
        lo, hi = covar_boot(xs, xi, 0.05, B=999, block=20, seed=SEED + i)
        iid = []
        for _ in range(999):
            idx = rng.integers(0, len(xi), len(xi))
            iid.append(covar_static(xs[idx], xi[idx], 0.05)['dcovar'])
        out[k] = {'b': c['b'], 'var_i': c['var_i'], 'covar': c['covar'], 'dcovar': c['dcovar'], 'lo': lo, 'hi': hi,
                  'iid_lo': np.percentile(iid, 2.5), 'iid_hi': np.percentile(iid, 97.5), 'var_sys': c['var_sys']}
    return out


def fig_b1(res):
    ks = list(res)
    xs = np.arange(len(ks))
    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    d = np.array([res[k]['dcovar'] for k in ks])
    ax.errorbar(xs - 0.12, d, yerr=[d - [res[k]['lo'] for k in ks], np.array([res[k]['hi'] for k in ks]) - d],
                fmt='o', color=MainBlue, capsize=3, lw=1, label='Moving-block bootstrap (blocks of 20 days)')
    ax.errorbar(xs + 0.12, d, yerr=[d - [res[k]['iid_lo'] for k in ks], np.array([res[k]['iid_hi'] for k in ks]) - d],
                fmt='s', color=Orange, capsize=3, lw=1, ms=4, label='i.i.d. bootstrap')
    ax.set_xticks(xs)
    ax.set_xticklabels([NAMES[k] for k in ks], fontsize=8, rotation=20, ha='right')
    ax.set_ylabel('ΔCoVaR 5% with 95% interval (%)')
    legend_outside_bottom(ax, ncol=2, y=-0.3)
    save_fig('ch18_sem_b1')


def b2_eu_dcovar():
    """Delta-CoVaR 1% si 5% pentru bancile europene si clasamentele lor."""
    out = {}
    for i, k in enumerate(EU):
        xs, xi = system_ex(k, EU).values, R[k].values
        c1, c5 = covar_static(xs, xi, 0.01), covar_static(xs, xi, 0.05)
        lo, hi = covar_boot(xs, xi, 0.01, B=999, block=20, seed=SEED + 10 + i)
        out[k] = {'d1': c1['dcovar'], 'd5': c5['dcovar'], 'lo1': lo, 'hi1': hi, 'var1': c1['var_i'], 'b1': c1['b']}
    t = pd.DataFrame(out).T
    return {'table': t.to_dict(orient='index'), 'rank1': list(t['d1'].sort_values(ascending=False).index),
            'rank5': list(t['d5'].sort_values(ascending=False).index),
            'spearman': stats.spearmanr(t['d1'], t['d5']).correlation}


def b4_rank_boot(B=999):
    """Corelatia Spearman intre VaR 1% si Delta-CoVaR 1% pe cele 11 banci, cu bootstrap pe blocuri pe zile."""
    ks = US + EU
    def stat(idx):
        v, d = [], []
        for k in ks:
            mem = US if k in US else EU
            xs = R[[j for j in mem if j != k]].mean(axis=1).values[idx]
            c = covar_static(xs, R[k].values[idx], 0.01)
            v.append(c['var_i'])
            d.append(c['dcovar'])
        return stats.spearmanr(v, d).correlation
    n = len(R)
    full = stat(np.arange(n))
    rng = np.random.default_rng(SEED)
    bs = [stat(block_bootstrap_idx(n, 20, rng)) for _ in range(B)]
    return {'rho': full, 'lo': np.percentile(bs, 2.5), 'hi': np.percentile(bs, 97.5), 'B': B}


def cholesky_fevd(A, S, H=10, order=None):
    """Descompunerea ortogonala (Cholesky) a dispersiei, pentru o ordine data a variabilelor."""
    k = S.shape[0]
    order = list(range(k)) if order is None else order
    P = np.eye(k)[order]
    Sp = P @ S @ P.T
    L = np.linalg.cholesky(Sp)
    Phi = [P @ F @ P.T for F in ma_coefs(A, H)]
    num = sum((F @ L) ** 2 for F in Phi)
    th = num / num.sum(axis=1, keepdims=True)
    inv = np.argsort(order)
    return th[np.ix_(inv, inv)]


def b5_dy_sensitivity():
    """Indicele total pentru orizonturi H si ordine VAR p; generalizat vs Cholesky in doua ordini."""
    Y = V.values
    grid = {}
    for p in [1, 2, 3, 4]:
        A, S = var_fit(Y, p)
        for H in [5, 10, 20]:
            grid[f'p{p}_H{H}'] = spillover_table(gfevd(A, S, H), ALL)['total']
    A, S = var_fit(Y, 1)
    ch_us = spillover_table(cholesky_fevd(A, S, 10), ALL)
    ch_ro = spillover_table(cholesky_fevd(A, S, 10, order=list(range(len(ALL)))[::-1]), ALL)
    g = spillover_table(gfevd(A, S, 10), ALL)
    return {'grid': grid, 'chol_us_first': ch_us['total'], 'chol_ro_first': ch_ro['total'], 'gen': g['total'],
            'net_us_first': ch_us['net'][US].sum(), 'net_ro_first': ch_ro['net'][US].sum(),
            'gen_net_us': g['net'][US].sum()}


def fig_b5(res):
    fig, ax = plt.subplots(figsize=(6.0, 2.8))
    for p, c in zip([1, 2, 3, 4], [MainBlue, IDAred, Forest, Purple]):
        ax.plot([5, 10, 20], [res['grid'][f'p{p}_H{H}'] for H in [5, 10, 20]], 'o-', color=c, label=f'VAR({p})')
    ax.set_xticks([5, 10, 20])
    ax.set_xlabel('Forecast horizon H (days)')
    ax.set_ylabel('Total connectedness (%)')
    legend_outside_bottom(ax, ncol=4, y=-0.25)
    save_fig('ch18_sem_b5')


def b6_rolling_test():
    """Media indicelui total 2020-2026 vs 2010-2019 (erori standard Newey-West pe seria cu pas de 5 zile)."""
    import statsmodels.api as sm
    tot = pd.read_csv(os.path.join(HERE, 'ch18_spill_rolling.csv'), index_col=0, parse_dates=True).iloc[:, 0]
    d = (tot.index >= '2020-01-01').astype(float)
    X = sm.add_constant(d)
    m = sm.OLS(tot.values, X).fit(cov_type='HAC', cov_kwds={'maxlags': 50})
    m0 = sm.OLS(tot.values, X).fit()
    return {'before': tot[tot.index < '2020-01-01'].mean(), 'after': tot[tot.index >= '2020-01-01'].mean(),
            'diff': m.params[1], 't_hac': m.tvalues[1], 't_ols': m0.tvalues[1], 'n': len(tot)}


def b6_subsample_boot(B=999):
    """Conectivitatea totala 2010-2019 fata de 2020-2026: VAR(1) pe fiecare subperioada, bootstrap pe blocuri de
    reziduuri in fiecare subperioada (extrageri independente) -> interval pentru diferenta."""
    Y1, Y2 = V.loc[:'2019-12-31'].values, V.loc['2020-01-01':].values
    t1, _ = spill_bootstrap(Y1, 1, B=B, seed=SEED + 61)
    t2, _ = spill_bootstrap(Y2, 1, B=B, seed=SEED + 62)
    c1 = spillover_table(gfevd(*var_fit(Y1, 1), 10), ALL)['total']
    c2 = spillover_table(gfevd(*var_fit(Y2, 1), 10), ALL)['total']
    d = t2 - t1
    return {'c1': c1, 'c2': c2, 'diff': c2 - c1, 'lo': np.percentile(d, 2.5), 'hi': np.percentile(d, 97.5),
            'lo1': np.percentile(t1, 2.5), 'hi1': np.percentile(t1, 97.5), 'lo2': np.percentile(t2, 2.5),
            'hi2': np.percentile(t2, 97.5), 'n1': len(Y1), 'n2': len(Y2), 'bias1': t1.mean() - c1, 'bias2': t2.mean() - c2}


def b7_frm_window():
    """FRM intr-o fereastra: lambda GACV si legaturile active pentru JPMorgan (calm vs COVID-19)."""
    D = frm_design()
    out = {}
    for end in ['2019-06-28', '2020-03-31']:
        w = D.loc[:end].iloc[-63:]
        r = frm_window(w, 'JPM')
        names = [j for j in ALL if j != 'JPM'] + ['S&P 500 (t-1)', 'Euro Stoxx 50 (t-1)', 'VIX (t-1)']
        coef = pd.Series(r['coef'], index=names)
        act = coef[coef.abs() > 1e-8].sort_values(key=np.abs, ascending=False)
        out[end] = {'lambda': r['lambda'], 'df': r['df_sel'], 'active': {NAMES.get(k, k): v for k, v in act.head(4).items()},
                    'n_active': len(act), 'first': w.index[0]}
    return out


def b8_ro_stress(value=1_000_000):
    """Portofoliu 50% Banca Transilvania / 50% BRD, rebalansat zilnic: scenarii istorice si un scenariu ipotetic pe factori.

    Randamentul log exact al portofoliului: 100 ln(1/2 e^{r_TLV/100} + 1/2 e^{r_BRD/100}) (nu media randamentelor log).
    Scenariul ipotetic: socurile factorilor sunt randamente log saptamanale; randamentul log prezis al fiecarei banci
    beta_k' f se transforma in randament simplu, apoi se aplica ponderile 50/50."""
    X = R[RO]
    port = 100 * np.log(np.exp(X / 100).mean(axis=1))
    out = {}
    for name, a, b in [('RO bank tax', '2018-12-10', '2019-01-31'), ('COVID-19', '2020-02-15', '2020-04-30')]:
        p = port.loc[a:b].rolling(10).sum()
        end = p.idxmin()
        i = port.index.get_loc(end)
        out[name] = {'loss': -100 * (np.exp(p.min() / 100) - 1), 'start': port.index[i - 9], 'end': end,
                     'loss_meanlog': -100 * (np.exp(X.mean(axis=1).loc[a:b].rolling(10).sum().min() / 100) - 1)}
        out[name]['amt'] = out[name]['loss'] / 100 * value
    W = weekly_returns(RO + ['SX5E', 'BET'])
    F = W[['SX5E', 'BET']]
    Xf = np.column_stack([np.ones(len(F)), F.values])
    coefs = {k: np.linalg.lstsq(Xf, W[k].values, rcond=None)[0] for k in RO}
    bet = np.mean([coefs[k][1:] for k in RO], axis=0)
    shock = np.array([-25.0, -20.0])                             # randamente log saptamanale (%)
    rhat = {k: coefs[k][1:] @ shock for k in RO}                 # randamentul log prezis al fiecarei banci (%)
    simple = {k: 100 * (np.exp(rhat[k] / 100) - 1) for k in RO}  # randamentul simplu corespunzator (%)
    loss = -np.mean([simple[k] for k in RO])                     # pierderea portofoliului 50/50 (%)
    out['hyp'] = {'b_sx5e': bet[0], 'b_bet': bet[1], 'loss': loss, 'amt': loss / 100 * value,
                  'loglin': -bet @ shock,
                  'b_bank': {k: list(coefs[k][1:]) for k in RO}, 'rhat': rhat, 'simple': simple,
                  'r2': np.mean([1 - np.var(W[k].values - Xf @ coefs[k]) / np.var(W[k].values) for k in RO])}
    return out


def c1_ro_exposure():
    """Expunerea bancilor romanesti la stresul bancar european: CoVaR de expunere, subperioade, conectivitate."""
    sys_eu = R[EU].mean(axis=1)
    out = {}
    for k in RO:
        row = {}
        for a in [0.05, 0.01]:
            e = covar_static(R[k], sys_eu, a)                   # banca romaneasca | sistemul european in criza
            lo, hi = covar_boot(R[k].values, sys_eu.values, a, B=999, block=20, seed=SEED + 30)
            row[f'e{int(a * 100)}'] = {'covar': e['covar'], 'dcovar': e['dcovar'], 'b': e['b'], 'var': -np.quantile(R[k], a),
                                       'lo': lo, 'hi': hi}
            dom = covar_static(R[k], R['BET'], a)
            row[f'bet{int(a * 100)}'] = {'dcovar': dom['dcovar'], 'b': dom['b']}
        for lab, a, b in [('p1', '2010-01-01', '2017-12-31'), ('p2', '2018-01-01', '2026-12-31')]:
            e = covar_static(R[k].loc[a:b], sys_eu.loc[a:b], 0.05)
            y_, x_ = R[k].loc[a:b].values, sys_eu.loc[a:b].values
            rng = np.random.default_rng(SEED + 31)
            dr = []
            for _ in range(999):
                idx = block_bootstrap_idx(len(y_), 20, rng)
                dr.append(covar_static(y_[idx], x_[idx], 0.05)['dcovar'])
            row[lab] = {'dcovar': e['dcovar'], 'b': e['b'], 'lo': np.percentile(dr, 2.5), 'hi': np.percentile(dr, 97.5),
                        'se': np.std(dr, ddof=1)}
        # subperioadele sunt disjuncte: extragerile bootstrap sunt independente, deci se(dif)^2 = se1^2 + se2^2
        dd = row['p2']['dcovar'] - row['p1']['dcovar']
        sd = np.sqrt(row['p1']['se'] ** 2 + row['p2']['se'] ** 2)
        row['diff'] = {'d': dd, 'se': sd, 'z': dd / sd, 'p': 2 * stats.norm.sf(abs(dd / sd))}
        row['corr'] = R[k].corr(sys_eu)
        row['corr_bet'] = R[k].corr(R['BET'])
        out[k] = row
    # conectivitatea (volatilitate): cat din dispersia bancilor romanesti vine de la bancile europene, pe ferestre mobile
    Y = V
    rows = {}
    for end in range(250, len(Y) + 1, 5):
        w = Y.iloc[end - 250:end]
        A, S = var_fit(w.values, 2)
        th = 100 * gfevd(A, S, 10)
        ro_i = [ALL.index(k) for k in RO]
        rows[Y.index[end - 1]] = {'EU': th[np.ix_(ro_i, [ALL.index(k) for k in EU])].sum(1).mean(),
                                  'US': th[np.ix_(ro_i, [ALL.index(k) for k in US])].sum(1).mean(),
                                  'RO_other': th[ro_i[0], ro_i[1]] / 2 + th[ro_i[1], ro_i[0]] / 2}
    f = pd.DataFrame(rows).T
    f.to_csv(os.path.join(HERE, 'ch18_ro_from_eu.csv'))
    out['roll'] = {'eu_mean': f['EU'].mean(), 'us_mean': f['US'].mean(), 'eu_max': f['EU'].max(),
                   'eu_max_date': f['EU'].idxmax(), 'eu_2019': f['EU'].loc['2019'].mean(),
                   'eu_2020': f['EU'].loc['2020-03':'2020-12'].mean(), 'eu_last': f['EU'].iloc[-1],
                   'eu_2022': f['EU'].loc['2022'].mean(), 'ro_other_mean': f['RO_other'].mean()}
    return out, f


def fig_c1(res, f):
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.9), gridspec_kw={'width_ratios': [1, 1.6]})
    ax = axes[0]
    xs = np.arange(2)
    for j, (lab, c) in enumerate([('p1', MainBlue), ('p2', IDAred)]):
        d = np.array([res[k][lab]['dcovar'] for k in RO])
        lo = np.array([res[k][lab]['lo'] for k in RO])
        hi = np.array([res[k][lab]['hi'] for k in RO])
        ax.bar(xs + (j - 0.5) * 0.36, d, 0.34, color=c, label='2010-2017' if lab == 'p1' else '2018-2026')
        ax.errorbar(xs + (j - 0.5) * 0.36, d, yerr=[d - lo, hi - d], fmt='none', ecolor='black', capsize=2, lw=0.8)
    ax.set_xticks(xs)
    ax.set_xticklabels([NAMES[k] for k in RO], fontsize=8)
    ax.set_ylabel('Exposure ΔCoVaR 5% (%)')
    legend_outside_bottom(ax, ncol=2, y=-0.18)
    ax = axes[1]
    ax.plot(f.index, f['EU'], color=IDAred, lw=0.9, label='From European banks')
    ax.plot(f.index, f['US'], color=MainBlue, lw=0.9, label='From US banks')
    ax.set_ylabel('Share of RO banks\' volatility\nforecast error variance (%)')
    ax.set_xlim(f.index[0], f.index[-1])
    mark_events(ax)
    legend_outside_bottom(ax, ncol=2, y=-0.18)
    fig.tight_layout()
    save_fig('ch18_sem_c1')


def fig_a1(s_sys=1.5, s_i=2.0, rho=0.6, alpha=0.01):
    """Densitatile conditionate ale sistemului: banca la mediana si banca la VaR 1%."""
    a = a1_covar_normal(s_sys, s_i, rho, alpha)
    z = a['z']
    sd = a['cond_sd']
    x = np.linspace(-9, 5, 600)
    fig, ax = plt.subplots(figsize=(6.2, 2.7))
    for mu, c, lab, q in [(0.0, MainBlue, 'System given bank at its median, X_i = 0', -a['covar_med']),
                          (rho * s_sys / s_i * s_i * z, IDAred, 'System given bank at its VaR 1%, X_i = q_1%', -a['covar'])]:
        ax.plot(x, stats.norm.pdf(x, mu, sd), color=c, lw=1.3, label=lab)
        xs = x[x <= q]
        ax.fill_between(xs, 0, stats.norm.pdf(xs, mu, sd), color=c, alpha=0.25)
        ax.axvline(q, color=c, lw=0.8, ls='--')
    y0 = 0.36
    ax.annotate('', xy=(-a['covar'], y0), xytext=(-a['covar_med'], y0),
                arrowprops=dict(arrowstyle='<->', color='black', lw=0.9))
    ax.text(-a['covar'] - 0.2, y0 + 0.02, f"ΔCoVaR 1% = {a['dcovar']:.2f}", ha='right',
            fontsize=8, color='black')
    ax.text(-a['covar'], 0.02, f"-CoVaR = {-a['covar']:.2f}", ha='right', fontsize=7.5, color=IDAred)
    ax.text(-a['covar_med'], 0.02, f" -CoVaR = -{a['covar_med']:.2f}", ha='left', fontsize=7.5, color=MainBlue)
    ax.set_xlabel('System return X_sys (%)')
    ax.set_ylabel('Conditional density')
    ax.set_ylim(0, 0.42)
    legend_outside_bottom(ax, ncol=1, y=-0.28)
    save_fig('ch18_sem_a1')


def fig_a7(b=(1.2, 0.8), S=((16, 6), (6, 9)), target=20):
    """Planul factorilor: elipsele Mahalanobis, dreapta de pierdere -b'f = l* si scenariul cel mai plauzibil."""
    b, S = np.asarray(b, float), np.asarray(S, float)
    r = reverse_2f(b, S, target)
    L = np.linalg.cholesky(S)
    t = np.linspace(0, 2 * np.pi, 400)
    fig, ax = plt.subplots(figsize=(4.6, 3.4))
    for c, col in [(1, Teal), (2, Forest), (r['d'], IDAred)]:
        e = (L @ np.vstack([np.cos(t), np.sin(t)])) * c
        ax.plot(e[0], e[1], color=col, lw=1.0, label=f'Mahalanobis distance {c:.2f}' if c > 2 else f'Mahalanobis distance {c:.0f}')
    f1 = np.linspace(-22, 6, 50)
    ax.plot(f1, (-target - b[0] * f1) / b[1], color=MainBlue, lw=1.2, label=f"Loss line -b'f = {target}%")
    ax.plot(*r['f'], 'o', color=IDAred, ms=6, label=f"f* = ({r['f'][0]:.1f}, {r['f'][1]:.1f})")
    ax.axhline(0, color=Gray, lw=0.5)
    ax.axvline(0, color=Gray, lw=0.5)
    ax.set_xlim(-22, 12)
    ax.set_ylim(-16, 12)
    ax.set_aspect('equal')
    ax.set_xlabel('Factor 1 return f1 (%)')
    ax.set_ylabel('Factor 2 return f2 (%)')
    legend_outside_bottom(ax, ncol=2, y=-0.22)
    save_fig('ch18_sem_a7')


def fig_b2(sem):
    """Delta-CoVaR 1% (interval bootstrap pe blocuri) si 5% pentru bancile europene; VaR 1% vs Delta-CoVaR 1%."""
    t = sem['B2']['table']
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.9), gridspec_kw={'width_ratios': [1.2, 1]})
    ax = axes[0]
    xs = np.arange(len(EU))
    d1 = np.array([t[k]['d1'] for k in EU])
    ax.errorbar(xs - 0.1, d1, yerr=[d1 - [t[k]['lo1'] for k in EU], np.array([t[k]['hi1'] for k in EU]) - d1],
                fmt='o', color=IDAred, capsize=3, lw=1, label='ΔCoVaR 1% with 95% block-bootstrap interval')
    ax.plot(xs + 0.1, [t[k]['d5'] for k in EU], 's', color=MainBlue, ms=5, label='ΔCoVaR 5%')
    ax.set_xticks(xs)
    ax.set_xticklabels([NAMES[k] for k in EU], fontsize=8, rotation=20, ha='right')
    ax.set_ylabel('ΔCoVaR (%)')
    legend_outside_bottom(ax, ncol=1, y=-0.32)
    ax = axes[1]
    for k in US + EU:
        col = MainBlue if k in US else IDAred
        c1 = covar_static(system_ex(k, US if k in US else EU).values, R[k].values, 0.01)
        ax.plot(c1['var_i'], c1['dcovar'], 'o', color=col, ms=5)
        ax.annotate(NAMES[k], (c1['var_i'], c1['dcovar']), fontsize=6, color='black',
                    xytext=(3, 2), textcoords='offset points')
    ax.plot([], [], 'o', color=MainBlue, label='US banks')
    ax.plot([], [], 'o', color=IDAred, label='European banks')
    b4 = sem['B4']
    ax.set_title(f"Spearman {b4['rho']:.2f}, 95% interval [{b4['lo']:.2f}, {b4['hi']:.2f}]", fontsize=8, color='black')
    ax.set_xlabel('VaR 1% of the bank (%)')
    ax.set_ylabel('ΔCoVaR 1% (%)')
    legend_outside_bottom(ax, ncol=2, y=-0.32)
    fig.tight_layout()
    save_fig('ch18_sem_b2')


def lrmes_paths(end, h, C, Sn, seed, want):
    """Aceeasi simulare ca inference18.lrmes_sim (aceleasi extrageri aleatoare), dar intoarce pierderea fiecarei
    banci cerute pe traiectoriile de criza (fractii), pentru graficele B3 si B4."""
    rng = np.random.default_rng(seed)
    Rs = R.loc[:end]
    out = {}
    for reg, mem in (('US', US), ('EU', EU), ('RO', RO)):
        rm = Rs[REGION_INDEX[reg]].values
        gm = gjr_fit(rm)
        eps_m = rm / gm['sig']
        T = len(rm)
        keep = []
        for _ in range(Sn // 20000):
            idx = rng.integers(0, T, size=(20000, h))
            cm = simulate_market(gm, eps_m, idx)
            keep.append(idx[np.exp(cm / 100) - 1 < C])
        kept = np.concatenate(keep)
        for k in mem:
            if k not in want:
                continue
            ri = Rs[k].values
            gf = gjr_fit(ri)
            e = np.column_stack([ri / gf['sig'], eps_m])
            dcc = dcc_fit(e)
            xi = (e[:, 0] - dcc['rho'] * e[:, 1]) / np.sqrt(1 - dcc['rho'] ** 2)
            ci = simulate_firm(gf, gm, dcc, eps_m, xi, kept)
            out[k] = -(np.exp(ci / 100) - 1)
    return out


def _loss_hist(ax, loss_today, loss_covid, lr_today, lr_covid, title):
    bins = np.linspace(-0.3, 0.8, 56)
    for L_, lr, c, lab in [(loss_today, lr_today, MainBlue, 'Information to 18 Sep 2026'),
                           (loss_covid, lr_covid, IDAred, 'Information to 31 Mar 2020')]:
        ax.hist(100 * L_, bins=100 * bins, density=True, histtype='step', color=c, lw=1.2,
                label=f'{lab}: LRMES {100 * lr:.1f}% (dashed), 90% range shaded')
        ax.axvline(100 * lr, color=c, lw=1.0, ls='--')
        q05, q95 = np.quantile(L_, [0.05, 0.95])
        ax.axvspan(100 * q05, 100 * q95, color=c, alpha=0.08)
    ax.set_title(title, fontsize=9, color='black')
    ax.set_xlabel('22-day loss of the bank on market-crisis paths (%)')
    ax.set_ylabel('Density')


def fig_b3_b4(inf=None):
    """Distributia pierderii bancii pe traiectoriile de criza (piata sub -10% in 22 de zile), azi si la 31.03.2020;
    cu inf (ch18_inference.json) se verifica egalitatea cu LRMES raportat."""
    end = R.index[-1]
    today = lrmes_paths(end, 22, -0.10, 200000, SEED, ['JPM', 'DBK', 'TLV'])
    covid = lrmes_paths(pd.Timestamp('2020-03-31'), 22, -0.10, 200000, SEED + 3, ['JPM', 'DBK', 'TLV'])
    for k in ['JPM', 'DBK', 'TLV']:
        if inf is not None:
            assert abs(today[k].mean() - inf['srisk']['be'][k]['lrmes']) < 1e-9, k
            assert abs(covid[k].mean() - inf['srisk']['covid'][k]['lrmes']) < 1e-9, k
    fig, ax = plt.subplots(figsize=(6.2, 2.8))
    _loss_hist(ax, today['JPM'], covid['JPM'], today['JPM'].mean(), covid['JPM'].mean(), 'JPMorgan, crisis: S&P 500 below -10% in 22 days')
    legend_outside_bottom(ax, ncol=1, y=-0.3)
    save_fig('ch18_sem_b3')
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.8))
    for ax, k, m in zip(axes, ['DBK', 'TLV'], ['Euro Stoxx 50', 'BET']):
        _loss_hist(ax, today[k], covid[k], today[k].mean(), covid[k].mean(), f'{NAMES[k]}, crisis: {m} below -10%')
    h, lab = axes[0].get_legend_handles_labels()
    fig.tight_layout()
    h2, lab2 = axes[1].get_legend_handles_labels()
    fig.legend(h + h2, [f'Deutsche Bank, {x}' for x in lab] + [f'Banca Transilvania, {x}' for x in lab2],
               loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=1, frameon=False, fontsize=7)
    save_fig('ch18_sem_b4')


def fig_b6(sem):
    """Indicele pe ferestre mobile cu mediile 2010-2019 si 2020-2026; estimarile pe subperioade cu intervale."""
    tot = pd.read_csv(os.path.join(HERE, 'ch18_spill_rolling.csv'), index_col=0, parse_dates=True).iloc[:, 0]
    b6, bs = sem['B6'], sem['B6S']
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.8), gridspec_kw={'width_ratios': [1.8, 1]})
    ax = axes[0]
    ax.plot(tot.index, tot.values, color=Purple, lw=0.8, label='Rolling total connectedness (250 days, VAR(2), H = 10)')
    cut = pd.Timestamp('2020-01-01')
    ax.hlines(b6['before'], tot.index[0], cut, color=MainBlue, lw=1.8, label=f"Mean 2010-2019: {b6['before']:.2f}%")
    ax.hlines(b6['after'], cut, tot.index[-1], color=IDAred, lw=1.8, label=f"Mean 2020-2026: {b6['after']:.2f}%")
    ax.axvline(cut, color=Gray, lw=0.6, ls=':')
    ax.set_xlim(tot.index[0], tot.index[-1])
    ax.set_ylabel('Total connectedness (%)')
    legend_outside_bottom(ax, ncol=1, y=-0.14)
    ax = axes[1]
    pts = [('2010-2019', bs['c1'], bs['lo1'], bs['hi1'], MainBlue), ('2020-2026', bs['c2'], bs['lo2'], bs['hi2'], IDAred)]
    for i, (lab, c, lo, hi, col) in enumerate(pts):
        ax.errorbar(i, c, yerr=[[c - lo], [hi - c]], fmt='o', color=col, capsize=4, lw=1.2)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['2010-2019', '2020-2026'])
    ax.set_xlim(-0.6, 1.6)
    ax.set_ylabel('C, VAR(1), H = 10 (%)')
    ax.set_title(f"Difference {bs['diff']:.1f}, 95% interval [{bs['lo']:.1f}, {bs['hi']:.1f}]", fontsize=8, color='black')
    fig.tight_layout()
    save_fig('ch18_sem_b6')


def fig_b7(sem):
    """Coeficientii regresiei cuantile penalizate pentru JPMorgan, fereastra calma vs fereastra de criza."""
    D = frm_design()
    names = [NAMES[j] for j in ALL if j != 'JPM'] + ['S&P 500 (t-1)', 'Euro Stoxx 50 (t-1)', 'VIX (t-1)']
    fig, ax = plt.subplots(figsize=(6.6, 3.0))
    ys = np.arange(len(names))
    for j, (end, c, lab) in enumerate([('2019-06-28', MainBlue, 'Calm window ending 28 Jun 2019'),
                                       ('2020-03-31', IDAred, 'COVID-19 window ending 31 Mar 2020')]):
        w = D.loc[:end].iloc[-63:]
        r = frm_window(w, 'JPM')
        assert abs(r['lambda'] - sem['B7'][end]['lambda']) < 1e-15 and r['df_sel'] == sem['B7'][end]['df']
        ax.barh(ys + (0.2 if j == 0 else -0.2), r['coef'], 0.38, color=c,
                label=f"{lab}: λ = {r['lambda']:.1e}, {r['df_sel']} of 15 active")
    ax.axvline(0, color=Gray, lw=0.6)
    ax.set_yticks(ys)
    ax.set_yticklabels(names, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel('Coefficient in the 5% quantile regression of JPMorgan')
    legend_outside_bottom(ax, ncol=1, y=-0.2)
    save_fig('ch18_sem_b7')


def fig_b8(sem):
    """Randamentul log cumulat al portofoliului 50/50 BT/BRD in cele doua ferestre, cu cele mai rele 10 zile."""
    X = R[RO]
    port = 100 * np.log(np.exp(X / 100).mean(axis=1))
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.7))
    for ax, (name, a, b, c) in zip(axes, [('RO bank tax', '2018-12-10', '2019-01-31', Forest),
                                          ('COVID-19', '2020-02-15', '2020-04-30', IDAred)]):
        p = port.loc[a:b]
        d = sem['B8'][name]
        ax.plot(p.index, p.cumsum(), color=c, lw=1.3, label=f'Cumulative log return of the portfolio, {name}')
        ax.axvspan(pd.Timestamp(d['start']), pd.Timestamp(d['end']), color=c, alpha=0.12,
                   label=f"Worst 10 days: loss {d['loss']:.1f}%")
        w = port.loc[d['start']:d['end']].sum()
        assert abs(-100 * (np.exp(w / 100) - 1) - d['loss']) < 1e-9
        ax.axhline(0, color=Gray, lw=0.5)
        ax.set_ylabel('Cumulative log return (%)')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=5))
        ax.tick_params(axis='x', labelsize=7)
        legend_outside_bottom(ax, ncol=1, y=-0.3)
    fig.tight_layout()
    save_fig('ch18_sem_b8')


def fig_b10(inf):
    """Evenimentele comune observate fata de regiunea de acceptare Kupiec; puterea testului."""
    bt = inf['backtest_ge']
    n, p0 = bt['JPM']['n'], 0.05 ** 2

    def lr(x, p):
        ph = x / n
        l1 = (x * np.log(ph) if x else 0) + (n - x) * np.log(1 - ph)
        return -2 * ((x * np.log(p) + (n - x) * np.log(1 - p)) - l1)

    rej = np.array([stats.chi2.sf(lr(x, p0), 1) < 0.05 for x in range(0, 80)])
    lo = max(x for x in range(0, int(n * p0)) if rej[x]) + 1
    hi = min(x for x in range(int(n * p0) + 1, 80) if rej[x]) - 1
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.6), gridspec_kw={'width_ratios': [1.4, 1]})
    fig.subplots_adjust(bottom=0.45, wspace=0.3)
    ax = axes[0]
    ks = US + EU
    ax.axhspan(lo - 0.5, hi + 0.5, color=Forest, alpha=0.12, label=f'Kupiec 5% acceptance region: {lo} to {hi} hits')
    ax.bar(np.arange(len(ks)), [bt[k]['hits'] for k in ks], color=[MainBlue if k in US else IDAred for k in ks],
           label='Observed joint hits')
    ax.axhline(n * p0, color='black', lw=0.9, ls='--', label=f'Expected under H0: {n * p0:.1f}')
    ax.set_xticks(np.arange(len(ks)))
    ax.set_xticklabels([NAMES[k] for k in ks], rotation=40, ha='right', fontsize=7)
    ax.set_ylabel('Joint hits')
    legend_outside_bottom(ax, ncol=1, y=-0.42)
    ax = axes[1]
    mult = np.linspace(1, 3, 41)
    xs = np.arange(0, 80)
    power = [stats.binom.pmf(xs[rej], n, m * p0).sum() for m in mult]
    ax.plot(mult, 100 * np.array(power), color=Purple, lw=1.4, label='Power of the 5% Kupiec test')
    p2 = 100 * stats.binom.pmf(xs[rej], n, 2 * p0).sum()
    ax.plot(2, p2, 'o', color=IDAred, label=f'Doubled rate: power {p2:.0f}%')
    ax.set_xlabel('True joint rate / α²')
    ax.set_ylabel('Rejection probability (%)')
    ax.set_ylim(0, 100)
    legend_outside_bottom(ax, ncol=1, y=-0.25)
    save_fig('ch18_sem_b10')

## Run

In [ ]:
for f in [a1_covar_normal, a2_ge_covar, a3_euler, a4_lrmes_shortcut, a5_gfevd, a6_gfevd, a7_reverse, a8_reverse]:
    print(f.__name__, f())
b1 = b1_us_dcovar5()
fig_b1(b1)
print(pd.DataFrame(b1).T.round(2))
b2 = b2_eu_dcovar()
print('B2', b2)
b4 = b4_rank_boot()
print('B4', b4)
b5 = b5_dy_sensitivity()
fig_b5(b5)
print('B5', b5)
tot, net = spill_rolling()
tot.to_csv(os.path.join(HERE, 'ch18_spill_rolling.csv'))
b6, b6s = b6_rolling_test(), b6_subsample_boot()
print('B6', b6, b6s)
b7 = b7_frm_window()
print('B7', b7)
b8 = b8_ro_stress()
print('B8', b8)
c1, f = c1_ro_exposure()
fig_c1(c1, f)
print('C1', c1)
sem = {'B2': b2, 'B4': b4, 'B6': b6, 'B6S': b6s, 'B7': b7, 'B8': b8}
fig_a1()
fig_a7()
fig_b2(sem)
fig_b6(sem)
fig_b7(sem)
fig_b8(sem)
fig_b10({'backtest_ge': backtest_ge()})
fig_b3_b4()   # two simulations of 200 000 paths

![ch18_sem_b1](./ch18_sem_b1.png)

Output: `ch18_sem_b1.pdf`

![ch18_sem_b5](./ch18_sem_b5.png)

Output: `ch18_sem_b5.pdf`

![ch18_sem_c1](./ch18_sem_c1.png)

Output: `ch18_sem_c1.pdf`

![ch18_sem_a1](./ch18_sem_a1.png)

Output: `ch18_sem_a1.pdf`

![ch18_sem_a7](./ch18_sem_a7.png)

Output: `ch18_sem_a7.pdf`

![ch18_sem_b2](./ch18_sem_b2.png)

Output: `ch18_sem_b2.pdf`

![ch18_sem_b3](./ch18_sem_b3.png)

Output: `ch18_sem_b3.pdf`

![ch18_sem_b4](./ch18_sem_b4.png)

Output: `ch18_sem_b4.pdf`

![ch18_sem_b6](./ch18_sem_b6.png)

Output: `ch18_sem_b6.pdf`

![ch18_sem_b7](./ch18_sem_b7.png)

Output: `ch18_sem_b7.pdf`

![ch18_sem_b8](./ch18_sem_b8.png)

Output: `ch18_sem_b8.pdf`

![ch18_sem_b10](./ch18_sem_b10.png)

Output: `ch18_sem_b10.pdf`